# 00 | Sample Selection

## Section 1 - Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u

In [ ]:
!pip install astroquery
from astroquery.sdss import SDSS

## Querying SDSS via astroquery

In [ ]:
query = """
SELECT TOP 3000
  p.objid, p.ra, p.dec, p.petroRad_r, s.z, s.class, p.modelMag_u, p.modelMag_g, p.modelMag_r, p.modelMag_i, p.modelMag_z, p.modelMagErr_u, p.modelMagErr_g, p.modelMagErr_r, p.modelMagErr_i, p.modelMagErr_z, s.snMedian
FROM PhotoObj AS p
JOIN SpecObj AS s ON p.objid = s.bestObjID
WHERE p.clean = 1 AND s.class = 'GALAXY' AND s.z BETWEEN 0.02 AND 0.25
"""

result = SDSS.query_sql(query)
result

In [ ]:
df = result.to_pandas()
df.head()

In [ ]:
df

## Quality Cuts

In [ ]:
cuts = (df['snMedian'] > 10) & (df['petroRad_r'] > 3) & (df['modelMag_r'] < 17.77)
df_clean = df[cuts]

In [ ]:
print(f"Before cuts: {len(df)}")
print(f"After cuts: {len(df_clean)}")
print(f"Fraction of targets retained: {len(df_clean)/len(df):.2f}")

## Plotting

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))

axes[0].hist(df_clean['z'])
axes[0].set_xlabel('Redshift')
axes[0].axvline(0.1, color='red', linestyle='--', label='low-z')
axes[0].legend()

axes[1].hist(df_clean['modelMag_r'])
axes[1].set_xlabel('r-band Magnitude')
axes[1].axvline(17.77, color='red', linestyle='--', label='Magnitude Cut')
axes[1].legend()


In [ ]:
df_clean['low_z'] = df_clean['z'] < 0.1
df_clean

In [ ]:
df_clean = df_clean.reset_index(drop=True)
df_clean

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(df['ra'], df['dec'], 'b.', label='Full Sample')

plt.plot(df_clean['ra'], df_clean['dec'], 'k.', label="After Cuts")
plt.xlabel('RA')
plt.ylabel('Dec')
plt.legend()

### Export panda dataframe to csv

In [ ]:
df_clean.to_csv("galaxy_sample.csv", index=False)